# Single-epoch regime (10-wire circuit)

What do the generalization experiments look like when every training sample
is seen **exactly once**? Task: 10-wire brickwork circuit, depth 4, circuit
seed 5, solo **wire 1** (depth-4 tap, gated in the final layer, depends on
all 10 inputs — verified below). `train_frac=0.5` gives a 512-input pool;
`data_order="epoch"` streams it in a shuffled order with no repetition, so
one epoch at batch 8 = **64 updates** on fresh data throughout.

Sections: circuit check -> LR sanity at w128d4 -> the single-epoch sweep
(7 shapes x noise in {0, 0.5, 1} x 3 seeds; runs take seconds) -> an
epochs dial at w128d4 (1 to 16 passes) to price what repetition buys.

Note `warmup=0` everywhere: the default 500-step warmup would outlast
these runs entirely. Per-shape LRs are the batch-256 tuned values —
directionally fine at batch 8, but the LR-check section keeps us honest.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import os
    %pip -q install -U "jax[cuda12]" optax
    if not os.path.exists("/content/circscale"):
        !git clone https://github.com/amdson/circscale.git /content/circscale
    %cd /content/circscale
    !git pull
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/circscale_runs", exist_ok=True)
    if not os.path.islink("runs"):
        os.symlink("/content/drive/MyDrive/circscale_runs", "runs")

## The circuit

In [ ]:
from itertools import product

import matplotlib.pyplot as plt
import numpy as np

from random_circuit import evaluate_np, sample_circuit
from train import RunConfig, load_run, run

N_WIRES, CIRC_DEPTH, CIRCUIT_SEED, TARGET_WIRE = 10, 4, 5, 1

circuit = sample_circuit(np.random.default_rng(CIRCUIT_SEED), N_WIRES, CIRC_DEPTH)
X = np.array(list(product([0, 1], repeat=N_WIRES)), dtype=np.uint8)
y = evaluate_np(circuit, X)[:, TARGET_WIRE]
gated4 = set()
for s in range(circuit.n_gate_slots):
    if not np.array_equal(circuit.tables[3, s], np.arange(8)):
        gated4 |= set(int(w) for w in circuit.perms[3, 3 * s:3 * s + 3])
flip = np.eye(N_WIRES, dtype=np.uint8)
powers = 1 << np.arange(N_WIRES)[::-1]
rel = [i for i in range(N_WIRES) if np.any(y != y[(X ^ flip[i]) @ powers])]
print(f"tap depths: {circuit.out_depths}")
print(f"wire {TARGET_WIRE}: depth {circuit.out_depths[TARGET_WIRE]}, "
      f"gated in final layer: {TARGET_WIRE in gated4}, "
      f"relevant inputs {len(rel)}/{N_WIRES}, balance {y.mean():.3f}")

## Config

In [ ]:
TRAIN_FRAC = 0.5          # pool of 512 of the 1024 inputs (pool_seed 0)
BATCH = 8
EPOCH_STEPS = int(TRAIN_FRAC * 2 ** N_WIRES) // BATCH   # 64 updates/epoch
WN_ARMS = [0.0, 0.5, 1.0]  # init-relative transient noise
SEEDS = [0, 1, 2]
OUT_DIR = "runs/ep10"
CHANCE = float(np.log(2))

# (width, mlp_depth, lr) — Adam LRs tuned at batch 256 (runs/lr_table.json)
SHAPES = [
    (32,  2, 1e-2),
    (64,  3, 1e-2),
    (128, 4, 3e-3),
    (180, 5, 3e-3),
    (256, 6, 1e-3),
    (360, 7, 1e-3),
    (512, 8, 1e-3),
]


def ep_cfg(width, depth, lr, epochs=1, **kw):
    return RunConfig(width=width, mlp_depth=depth, lr=lr, warmup=0,
                     batch=BATCH, steps=epochs * EPOCH_STEPS, eval_every=1,
                     n_wires=N_WIRES, circ_depth=CIRC_DEPTH,
                     circuit_seed=CIRCUIT_SEED, output_wires=(TARGET_WIRE,),
                     train_frac=TRAIN_FRAC, data_order="epoch",
                     out_dir=OUT_DIR, **kw)


def n_params(w, d, hr=4):
    h = hr * w
    return N_WIRES * w + d * (w + w * h + h * w) + w + w * N_WIRES


def final(d, key="per_out_acc_ho"):
    return d[key][-1, TARGET_WIRE]


print(f"one epoch = {EPOCH_STEPS} steps of batch {BATCH}")

## LR sanity check (w128d4, one epoch)

The tuned LRs come from batch-256 runs; check the neighborhood at batch 8
before trusting the sweep.

In [ ]:
for lr in (1e-3, 3e-3, 1e-2):
    accs = []
    for s in SEEDS:
        cfg = ep_cfg(128, 4, lr, model_seed=s)
        run(cfg, quiet=True)
        accs.append(final(load_run(cfg.npz_path)[1]))
    print(f"lr={lr:g}: final held-out acc {np.mean(accs):.3f} "
          f"(seeds: {' '.join(f'{a:.3f}' for a in accs)})")

## Single-epoch sweep: 7 shapes x noise x 3 seeds

Every run is one pass over the 512-sample pool (64 updates). Idempotent.

In [ ]:
sweep = {}
for w, d, lr in SHAPES:
    for wn in WN_ARMS:
        for s in SEEDS:
            cfg = ep_cfg(w, d, lr, weight_noise=wn, model_seed=s)
            run(cfg, quiet=True)
            sweep[(w, d, wn, s)] = load_run(cfg.npz_path)[1]
print(f"{len(sweep)} runs done")

## Trajectories and scaling

Panels: one per shape; color = noise level; thin lines = seeds. X-axis is
samples seen (= step x batch; each sample is new). Right-most panel: final
held-out accuracy vs params.

In [ ]:
wn_col = dict(zip(WN_ARMS, plt.cm.viridis(np.linspace(0, 0.85, len(WN_ARMS)))))
fig, axes = plt.subplots(2, 4, figsize=(17, 7), sharex=True, sharey=True)
for ax, (w, d, lr) in zip(axes.flat, SHAPES):
    for wn in WN_ARMS:
        for i, s in enumerate(SEEDS):
            r = sweep[(w, d, wn, s)]
            m = r["eval_steps"] > 0
            ax.plot(r["eval_steps"][m] * BATCH,
                    np.maximum(r["per_out_loss_ho"][m][:, TARGET_WIRE], 1e-6),
                    color=wn_col[wn], lw=1.0, alpha=0.6,
                    label=f"wn={wn:g}" if i == 0 else None)
    ax.axhline(CHANCE, color="gray", ls=":", lw=0.8)
    ax.set(xscale="log", yscale="log", title=f"w{w}d{d}")
axes.flat[0].legend(fontsize=7)
axes.flat[0].set(ylabel="held-out BCE")

ax = axes.flat[7]
for wn in WN_ARMS:
    Ns = [n_params(w, d) for w, d, lr in SHAPES]
    means = [np.mean([final(sweep[(w, d, wn, s)]) for s in SEEDS])
             for w, d, lr in SHAPES]
    for (w, d, lr), N in zip(SHAPES, Ns):
        ax.plot([N] * len(SEEDS),
                [final(sweep[(w, d, wn, s)]) for s in SEEDS],
                "o", color=wn_col[wn], ms=3, alpha=0.4)
    ax.plot(Ns, means, "-o", color=wn_col[wn], ms=4, label=f"wn={wn:g}")
ax.axhline(0.5, color="gray", ls=":", lw=0.8)
ax.set(xscale="log", yscale="linear", xlabel="params N",
       ylabel="final held-out acc", title="after one epoch")
ax.legend(fontsize=7)
fig.supxlabel("samples seen")
plt.tight_layout()

## The epochs dial (w128d4)

Same pool, 1 to 16 passes (reshuffled each epoch). What does repetition buy
over the single pass, and does noise change that?

In [ ]:
EPOCHS = [1, 2, 4, 8, 16]
dial = {}
for e in EPOCHS:
    for wn in (0.0, 0.5):
        for s in SEEDS:
            cfg = ep_cfg(128, 4, 3e-3, epochs=e, weight_noise=wn, model_seed=s)
            run(cfg, quiet=True)
            dial[(e, wn, s)] = load_run(cfg.npz_path)[1]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.4))
for wn, col in ((0.0, "C0"), (0.5, "C1")):
    for e in EPOCHS:
        ax1.plot([e] * len(SEEDS), [final(dial[(e, wn, s)]) for s in SEEDS],
                 "o", color=col, ms=4, alpha=0.4)
    ax1.plot(EPOCHS, [np.mean([final(dial[(e, wn, s)]) for s in SEEDS])
                      for e in EPOCHS], "-o", color=col, label=f"wn={wn:g}")
    for i, s in enumerate(SEEDS):
        r = dial[(EPOCHS[-1], wn, s)]
        m = r["eval_steps"] > 0
        ax2.plot(r["eval_steps"][m] * BATCH,
                 np.maximum(r["per_out_loss_ho"][m][:, TARGET_WIRE], 1e-6),
                 color=col, lw=1.0, alpha=0.6,
                 label=f"wn={wn:g} ({EPOCHS[-1]} epochs)" if i == 0 else None)
ax1.axhline(0.5, color="gray", ls=":", lw=0.8)
ax1.set(xscale="log", xlabel="epochs over the 512-sample pool",
        ylabel="final held-out acc")
ax1.set_xticks(EPOCHS, labels=[str(e) for e in EPOCHS])
ax1.legend(fontsize=8)
ax2.axhline(CHANCE, color="gray", ls=":", lw=0.8)
ax2.set(xscale="log", yscale="log", xlabel="samples seen (with repetition)",
        ylabel="held-out BCE")
ax2.legend(fontsize=8)
plt.tight_layout()

## Notes

- One epoch at batch 8 = 64 updates; there is no train/eval distinction in
  time (every batch is unseen data), so the train-pool vs held-out gap only
  opens with repetition — compare the epochs dial.
- `data_order="epoch"`: fresh shuffle per epoch, each sample exactly once
  per pass; `batch` must divide the pool (8 | 512).
- 2^10 inputs means held-out accuracy has granularity 1/512.
- LRs are batch-256 tuned; rerun the LR-check cell before trusting a shape
  if its curves look pathological. Batch 8 also makes gradients 32x
  noisier — minibatch noise partially substitutes for weight noise here,
  which is itself part of what this regime measures.
- Runs are tiny (seconds); JIT compilation dominates wall time.